# Dataset Upload, Extraction & Directory Tree Inspector

A Google Colab notebook to:
1. Upload a `.zip` dataset directly from your computer (no Google Drive needed)
2. Extract it safely into `./dataset`
3. Print a clean, visual directory tree so you can inspect the structure

**Tip:** Run the cells top to bottom. For best speed on large datasets, use a GPU/CPU runtime as needed. Files uploaded to Colab are temporary and are deleted when the runtime disconnects.

## 1. Upload the Dataset (.zip)

Click **Choose Files** when the widget appears and select your `.zip` file.

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nUploaded files:")
for name, data in uploaded.items():
    print(f"  - {name}  ({len(data) / (1024 ** 2):.2f} MB)")

## 2. Extract the Dataset

This cell will:
- Automatically detect the uploaded `.zip` file
- Extract it into `EXTRACT_DIR` (with a path-traversal safety check)
- Remove junk files (`__MACOSX`, `.DS_Store`, `Thumbs.db`)
- Delete the original `.zip` to free disk space

Adjust the settings at the top of the cell if needed.

In [ ]:
import shutil
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

# ----------------------------- Settings -----------------------------
EXTRACT_DIR = Path("./dataset")   # designated working directory
DELETE_ZIP_AFTER = True           # remove the .zip after extraction
CLEAN_JUNK = True                 # remove __MACOSX, .DS_Store, Thumbs.db
OVERWRITE = True                  # wipe EXTRACT_DIR first if it exists
# --------------------------------------------------------------------

# 1) Detect the uploaded zip file(s)
try:
    zip_candidates = [Path(n) for n in uploaded if n.lower().endswith(".zip")]
except NameError:
    zip_candidates = []

if not zip_candidates:  # fallback: look in the current working directory
    zip_candidates = sorted(Path(".").glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)

if not zip_candidates:
    raise FileNotFoundError("No .zip file found. Please run the upload cell and select a .zip file.")

zip_path = zip_candidates[0]
if len(zip_candidates) > 1:
    print(f"Multiple zip files found; using the first one: {zip_path.name}")
print(f"Detected zip file: {zip_path}  ({zip_path.stat().st_size / (1024 ** 2):.2f} MB)")

if not zipfile.is_zipfile(zip_path):
    raise zipfile.BadZipFile(f"'{zip_path}' is not a valid zip archive.")

# 2) Prepare the destination folder
if EXTRACT_DIR.exists() and OVERWRITE:
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
dest_root = EXTRACT_DIR.resolve()

# 3) Extract with a safety check against path traversal ("zip slip")
with zipfile.ZipFile(zip_path) as zf:
    members = zf.infolist()
    for member in tqdm(members, desc="Extracting", unit="file"):
        target = (dest_root / member.filename).resolve()
        if dest_root != target and dest_root not in target.parents:
            raise RuntimeError(f"Unsafe path detected in zip: {member.filename}")
        zf.extract(member, dest_root)

print(f"\nExtracted {len(members)} entries into: {dest_root}")

# 4) Cleanup
if CLEAN_JUNK:
    removed = 0
    for junk_dir in list(EXTRACT_DIR.rglob("__MACOSX")):
        shutil.rmtree(junk_dir, ignore_errors=True)
        removed += 1
    for pattern in (".DS_Store", "Thumbs.db"):
        for junk_file in list(EXTRACT_DIR.rglob(pattern)):
            junk_file.unlink(missing_ok=True)
            removed += 1
    print(f"Removed {removed} junk item(s).")

if DELETE_ZIP_AFTER:
    zip_path.unlink(missing_ok=True)
    print(f"Deleted original archive: {zip_path.name}")

print("\nDone. Dataset is ready at:", dest_root)

## 3. Directory Tree Generator

A recursive `pathlib`-based tree printer. Parameters:

| Parameter | Description |
|---|---|
| `root` | Folder to inspect |
| `max_depth` | How many levels deep to print (root = level 0) |
| `max_entries` | Max items shown per folder (the rest are summarised as `... N more`). Useful for image datasets with thousands of files |
| `show_files` | Set `False` to show folders only |
| `show_hidden` | Include dotfiles/dotfolders |
| `show_size` | Show file sizes |

In [ ]:
from pathlib import Path


def _human_size(num_bytes: int) -> str:
    size = float(num_bytes)
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if size < 1024 or unit == "TB":
            return f"{size:.0f} {unit}" if unit == "B" else f"{size:.1f} {unit}"
        size /= 1024


def print_tree(
    root,
    max_depth: int = 3,
    max_entries: int = 10,
    show_files: bool = True,
    show_hidden: bool = False,
    show_size: bool = False,
):
    """Print a visual directory tree and return (n_dirs, n_files) for the printed portion."""
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"{root} does not exist.")

    counts = {"dirs": 0, "files": 0}

    def _list_children(directory: Path):
        try:
            children = list(directory.iterdir())
        except PermissionError:
            return None
        if not show_hidden:
            children = [c for c in children if not c.name.startswith(".")]
        if not show_files:
            children = [c for c in children if c.is_dir()]
        # folders first, then files; both alphabetical (case-insensitive)
        return sorted(children, key=lambda c: (not c.is_dir(), c.name.lower()))

    def _walk(directory: Path, prefix: str, depth: int):
        if depth > max_depth:
            return
        children = _list_children(directory)
        if children is None:
            print(f"{prefix}[permission denied]")
            return

        shown = children[:max_entries] if max_entries else children
        hidden_count = len(children) - len(shown)

        for i, child in enumerate(shown):
            is_last = (i == len(shown) - 1) and hidden_count == 0
            connector = "└── " if is_last else "├── "
            extension = "    " if is_last else "│   "

            if child.is_dir():
                counts["dirs"] += 1
                n_items = sum(1 for _ in child.iterdir()) if depth == max_depth else None
                suffix = f"  ({n_items} items, not expanded)" if n_items else ""
                print(f"{prefix}{connector}📁 {child.name}/{suffix}")
                _walk(child, prefix + extension, depth + 1)
            else:
                counts["files"] += 1
                size_txt = f"  [{_human_size(child.stat().st_size)}]" if show_size else ""
                print(f"{prefix}{connector}📄 {child.name}{size_txt}")

        if hidden_count > 0:
            print(f"{prefix}└── ... {hidden_count} more item(s)")

    print(f"📂 {root.resolve()}/")
    _walk(root, "", 1)
    print(f"\nShown: {counts['dirs']} folder(s), {counts['files']} file(s)  "
          f"(max_depth={max_depth}, max_entries={max_entries})")
    return counts["dirs"], counts["files"]

### Run the tree

Change `max_depth` and `max_entries` to explore the dataset at different levels of detail.

In [ ]:
print_tree(
    "./dataset",
    max_depth=3,       # how deep to go
    max_entries=8,     # max items displayed per folder
    show_files=True,
    show_hidden=False,
    show_size=False,
)

### (Optional) Full dataset summary

Counts **all** files by extension and total size, regardless of how many were displayed in the tree above.

In [ ]:
from collections import Counter
from pathlib import Path

root = Path("./dataset")
ext_counter = Counter()
total_bytes = 0
total_files = 0

for p in root.rglob("*"):
    if p.is_file():
        total_files += 1
        total_bytes += p.stat().st_size
        ext_counter[p.suffix.lower() or "(no extension)"] += 1

print(f"Total files : {total_files}")
print(f"Total size  : {_human_size(total_bytes)}\n")
print(f"{'Extension':<18}{'Count':>10}")
print("-" * 28)
for ext, n in ext_counter.most_common():
    print(f"{ext:<18}{n:>10}")